# Analyzing Files

In this notebook, we send PDF files to an OpenAI model and ask whether its an invoice. If that's the case, we extract fields like the vendor and the payment amount from it. 

Docs:
- [Analyze images and files](https://developers.openai.com/api/docs/quickstart?input-type=image-url#analyze-images-and-files)
- [File inputs](https://developers.openai.com/api/docs/guides/file-inputs)

## Setup

In [1]:
import base64
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(override=True)
client = OpenAI()

## Different options



The PDF is read from disk and encoded as base64.

In [2]:
INVOICE_PATH = Path("files/Anthropic_Invoice.pdf")
invoice_base64 = base64.b64encode(INVOICE_PATH.read_bytes()).decode("utf-8")

## Defining the model

In [3]:
MODEL = "gpt-5.4-mini"

## Define the data we want back

A Pydantic model gives the result a clear shape. The model first classifies whether the file is really an invoice. The invoice fields can be `None` when the file is not an invoice or when a value is missing.

In [4]:
class DocumentAnalysis(BaseModel):
    is_invoice: bool
    vendor: str | None
    payment_date: str | None
    amount: float | None
    currency: str | None

# Define Instructions

In [5]:
INSTRUCTIONS = (
    "Decide whether the document is really an invoice. "
    "If it is not an invoice, set is_invoice to false and set "
    "vendor, payment_date, amount, and currency to null. "
    "If it is an invoice, set is_invoice to true and extract "
    "the invoice vendor, payment date, total amount, and currency. "
    "Use YYYY-MM-DD for the payment date. "
    "Use null for any missing invoice field."
)

## Option 1 - Web URL

In [7]:
response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_url": "https://www.berkshirehathaway.com/letters/2024ltr.pdf",
                },
            ],
        },
    ],
    text_format=DocumentAnalysis
)

document_analysis = response.output_parsed

if document_analysis is not None:
    print(document_analysis.model_dump_json(indent=2))

{
  "is_invoice": false,
  "vendor": null,
  "payment_date": null,
  "amount": null,
  "currency": null
}


## Option 2 - Send base64 encoded file

The base64 encoded PDF is sent as an `input_file`. The extraction rules go into `instructions`, because they are stable app instructions.

In [ ]:
response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": INVOICE_PATH.name,
                    "file_data": f"data:application/pdf;base64,{invoice_base64}",
                },
            ],
        }
    ],
    text_format=DocumentAnalysis,
)

document_analysis = response.output_parsed

if document_analysis is not None:
    print(document_analysis.model_dump_json(indent=2))

## Option 3 - File Upload

In case you need to make multiple calls for a file, you can upload files first, and then reference them with the file id. 

`client.files.create(...)` reads the local file, uploads it to OpenAI, and returns a file object with an id you can use later. 

`"rb"` means read binary, so the PDF is opened as raw bytes for uploading.

In [10]:

print("Start uploading the file to the OpenAI API...")
file = client.files.create(
    file=open(INVOICE_PATH, "rb"),
    purpose="user_data"
)

print("Upload completed. File ID:", file.id)

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_id": file.id,
                },
            ],
        },
    ],
    text_format=DocumentAnalysis
)

document_analysis = response.output_parsed

if document_analysis is not None:
    print(document_analysis.model_dump_json(indent=2))

Start uploading the file to the OpenAI API...
Upload completed. File ID: file-7bGvqnmoYEoomksJmmQd28
{
  "is_invoice": true,
  "vendor": "Anthropic Ireland, Limited",
  "payment_date": "2026-05-09",
  "amount": 18.0,
  "currency": "EUR"
}


# Extract Data locally?

To save costs, it often makes sense to use a library that converts a PDF to markdown and only send the markdown to the LLM instead of a large PDF. 

Some popular PDF => Markdown conversation libraries: 
- [MinerU](https://github.com/opendatalab/MinerU)
- [Docling](https://docling-project.github.io/docling/)
- [Marker](https://github.com/datalab-to/marker)
- [PyMuPDF4LLM](https://github.com/pymupdf/pymupdf4llm)

Here we will use PyMuPDF4LLM because its the most lightweight solution. Docling, Marker, and MinerU are more powerful document-understanding systems, but they bring substantially more dependencies and possible model/runtime overhead.
